In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# **Download Data**

In [ ]:
# download uploaded embeddings, labels and file names
!gdown 1L1CDOKDRW1tSReHCEibHw_nS7Vwi_1IM
!gdown 1M-pMev1QbH4Hc2Y7NcvMSmP_xNePJAPv
!gdown 1HSKrHda-Jo2YgZkDltrfbaXQCoormyCN
!gdown 1w2b9EDjOBGTGxzBISKCT7KMTNgl87CFn
!gdown 1idOF4EvANNd2dBBCe3TkKW2pxibo2S2N
!gdown 1-6wXkicrOrbBnoO6kF8huJEVO5hrrXMk
!gdown 1AyFyBNk4IdkA_HhDuyDg3ZfiSD4XbMTG
!gdown 15T6Bq_VppISFO03NAGU8iGm1ZsTxtUl9
!gdown 1IS5duZ90QArgK6d1xFlKrK9zIov_koh6
!gdown 1vjgUiHowQQsUtd4-la5MeL_1-JmPnlNx
!gdown 1b1KFG4d9vhWuCbAT7-1ztgxndSIw8xXM

Downloading...
From: https://drive.google.com/uc?id=1L1CDOKDRW1tSReHCEibHw_nS7Vwi_1IM
To: /content/dino_test_embeddings.pth
100% 4.61M/4.61M [00:00<00:00, 34.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1M-pMev1QbH4Hc2Y7NcvMSmP_xNePJAPv
To: /content/dino_test_labels.pth
100% 25.2k/25.2k [00:00<00:00, 78.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1HSKrHda-Jo2YgZkDltrfbaXQCoormyCN
To: /content/dino_train_and_test_file_names.json
100% 795k/795k [00:00<00:00, 41.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1w2b9EDjOBGTGxzBISKCT7KMTNgl87CFn
To: /content/dino_train_embeddings.pth
100% 45.7M/45.7M [00:00<00:00, 55.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1idOF4EvANNd2dBBCe3TkKW2pxibo2S2N
To: /content/dino_train_labels.pth
100% 239k/239k [00:00<00:00, 96.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-6wXkicrOrbBnoO6kF8huJEVO5hrrXMk
To: /content/sup_test_embeddings.npy
100% 6.14M/6.14M [00:00<00:00, 70.5MB/s]
Downloadin

In [ ]:
import torch
import numpy as np
import json
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
import csv

In [ ]:
# dino train stuffs
dino_train_embeddings = torch.load("./dino_train_embeddings.pth")
dino_train_labels = torch.load("./dino_train_labels.pth")
with open('dino_train_and_test_file_names.json') as f:
  dino_train_file_names = json.load(f)['train_file_names']

# dino test stuffs
dino_test_embeddings = torch.load("./dino_test_embeddings.pth")
dino_test_labels = torch.load("./dino_test_labels.pth")
with open('dino_train_and_test_file_names.json') as f:
  dino_test_file_names = json.load(f)['test_file_names']

# sup train stuffs
supervised_train_embeddings = np.load("./sup_train_embeddings.npy")
supervised_train_labels = np.load("./sup_train_labels.npy")
with open('sup_train_file_names.json') as f:
  supervised_train_file_names = json.load(f)['test_file_names']

# sup test stuffs
supervised_test_embeddings = np.load("./sup_test_embeddings.npy")
supervised_test_labels = np.load("./sup_test_labels.npy")
with open('sup_test_file_names.json') as f:
  supervised_test_file_names = json.load(f)['test_file_names']

In [ ]:
# dino train stuffs
dino_train_embeddings = np.asarray(dino_train_embeddings)
dino_train_labels = np.asarray(dino_train_labels)
dino_train_file_names = dino_train_file_names

# dino test stuffs
dino_test_embeddings = np.asarray(dino_test_embeddings)
dino_test_labels = np.asarray(dino_test_labels)
dino_test_file_names = dino_test_file_names

# supervised train stuffs
supervised_train_embeddings = np.asarray(supervised_train_embeddings)
supervised_train_labels = np.asarray(supervised_train_labels)
for i, name in enumerate(supervised_train_file_names):
  supervised_train_file_names[i] = (name.split("/")[-1])[2:]

# supervised test stuffs
supervised_test_embeddings = np.asarray(supervised_test_embeddings)
supervised_test_labels = np.asarray(supervised_test_labels)
for i, name in enumerate(supervised_test_file_names):
  supervised_test_file_names[i] = (name.split("/")[-1])[2:]

# **Create CSV file**

In [ ]:
number_of_selected_pairs = 5

In [ ]:
csv_tuple = list()

for i in tqdm(range(len(dino_train_file_names))):
  temp_csv_tuple = list()

  # dino
  current_dino_embedding = dino_train_embeddings[i, :]
  current_dino_label = int(dino_train_labels[i])
  current_dino_file_name = dino_train_file_names[i]
  current_dino_patient_id = int(current_dino_file_name.split(".")[0].split("_")[-1])

  dino_knn = NearestNeighbors(n_neighbors=len(dino_train_embeddings), metric="euclidean")
  dino_knn.fit(dino_train_embeddings)
  _, dino_indices = dino_knn.kneighbors([current_dino_embedding])
  dino_sorted_indices = dino_indices.flatten()
  dino_sorted_indices = dino_sorted_indices[1:]

  positive_pair_counter = 0
  negative_pair_counter = 0
  for index in dino_sorted_indices:
    neighbor_embedding = dino_train_embeddings[index, :]
    neighbor_label = int(dino_train_labels[index])
    neighbor_file_name = dino_train_file_names[index]
    neighbor_patient_id = int(neighbor_file_name.split(".")[0].split("_")[-1])
    is_same_patient = current_dino_patient_id == neighbor_patient_id
    is_same_label = 1 if current_dino_label == neighbor_label else 0
    if not is_same_patient and is_same_label == 1 and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and positive_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, is_same_label))
      positive_pair_counter += 1
    elif not is_same_patient and is_same_label == 0 and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and negative_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, is_same_label))
      negative_pair_counter += 1
    elif is_same_patient and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and negative_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, 0))
      negative_pair_counter += 1
    if positive_pair_counter >= number_of_selected_pairs and negative_pair_counter >= number_of_selected_pairs:
      break

  # supervised
  supervised_index = supervised_train_file_names.index(current_dino_file_name)
  current_supervised_embedding = supervised_train_embeddings[supervised_index, :]
  current_supervised_label = int(supervised_train_labels[supervised_index])
  current_supervised_file_name = supervised_train_file_names[supervised_index]
  current_supervised_patient_id = int(current_supervised_file_name.split(".")[0].split("_")[-1])

  supervised_knn = NearestNeighbors(n_neighbors=len(supervised_train_embeddings), metric="euclidean")
  supervised_knn.fit(supervised_train_embeddings)
  _, supervised_indices = supervised_knn.kneighbors([current_supervised_embedding])
  supervised_sorted_indices = supervised_indices.flatten()
  supervised_sorted_indices = supervised_sorted_indices[1:]

  positive_pair_counter = 0
  negative_pair_counter = 0
  for index in supervised_sorted_indices:
    neighbor_file_name = supervised_train_file_names[index]
    dino_index = dino_train_file_names.index(neighbor_file_name)
    neighbor_embedding = dino_train_embeddings[dino_index, :]
    neighbor_label = int(dino_train_labels[dino_index])
    neighbor_patient_id = int(neighbor_file_name.split(".")[0].split("_")[-1])
    is_same_patient = current_dino_patient_id == neighbor_patient_id
    is_same_label = 1 if current_dino_label == neighbor_label else 0
    if not is_same_patient and is_same_label == 1 and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and positive_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, is_same_label))
      positive_pair_counter += 1
    elif not is_same_patient and is_same_label == 0 and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and negative_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, is_same_label))
      negative_pair_counter += 1
    elif is_same_patient and not (current_dino_file_name, neighbor_file_name, is_same_label) in temp_csv_tuple and negative_pair_counter < number_of_selected_pairs:
      temp_csv_tuple.append((current_dino_file_name, neighbor_file_name, 0))
      negative_pair_counter += 1
    if positive_pair_counter >= number_of_selected_pairs and negative_pair_counter >= number_of_selected_pairs:
      break

  csv_tuple += temp_csv_tuple

100%|██████████| 29732/29732 [1:08:13<00:00,  7.26it/s]


In [ ]:
len(csv_tuple)

594640

In [ ]:
csv_file_path = f"csv_for_k_{number_of_selected_pairs}.csv"
with open(csv_file_path, mode="w", newline='') as file:
  writer = csv.writer(file)
  writer.writerows(csv_tuple)

In [ ]:
!head -20 ./csv_for_k_5.csv

s0_10000_s0_18_33.png,s0_9989_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_8812_s0_16_33.png,0
s0_10000_s0_18_33.png,s0_10010_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_9458_s0_17_33.png,0
s0_10000_s0_18_33.png,s0_8827_s0_16_33.png,0
s0_10000_s0_18_33.png,s0_2304_s0_0_36.png,1
s0_10000_s0_18_33.png,s0_837_s0_0_132.png,1
s0_10000_s0_18_33.png,s0_2397_s0_0_36.png,1
s0_10000_s0_18_33.png,s0_2347_s0_0_36.png,1
s0_10000_s0_18_33.png,s0_5685_s0_12_104.png,1
s0_10000_s0_18_33.png,s0_9980_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_10017_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_6919_s0_13_33.png,0
s0_10000_s0_18_33.png,s0_9973_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_9975_s0_18_33.png,0
s0_10000_s0_18_33.png,s0_5811_s0_12_109.png,1
s0_10000_s0_18_33.png,s0_2156_s0_0_32.png,1
s0_10000_s0_18_33.png,s0_1981_s0_0_32.png,1
s0_10000_s0_18_33.png,s0_2151_s0_0_32.png,1
s0_10000_s0_18_33.png,s0_2164_s0_0_32.png,1


In [ ]:
!cp ./csv_for_k_5.csv ./gdrive/MyDrive/CrossEncoderEmbeddings/